In [1]:
import numpy as np
import pandas as pd, os, datetime

import matplotlib.pyplot as plt

# Import the loader function
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
from process_code import load_generation_data

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-26.01/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that i

Dask dashboard: /proxy/8787/status


2026-01-14 12:30:48,426 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 303c94ee39429c3f2da4aa497ec99c5b initialized by task ('shuffle-transfer-303c94ee39429c3f2da4aa497ec99c5b', 2) executed on worker tcp://127.0.0.1:42643
2026-01-14 12:30:55,432 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 89e9509ce1489bcd80a5aabb8e796b05 initialized by task ('shuffle-transfer-89e9509ce1489bcd80a5aabb8e796b05', 0) executed on worker tcp://127.0.0.1:33245
2026-01-14 12:30:57,779 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 303c94ee39429c3f2da4aa497ec99c5b deactivated due to stimulus 'task-finished-1768354257.7747786'
2026-01-14 12:30:59,140 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 89e9509ce1489bcd80a5aabb8e796b05 deactivated due to stimulus 'task-finished-1768354258.9565947'


In [2]:
data, info = load_generation_data(
    sdate="2009-07-01",
    edate="2024-06-30",
    mode="daily",
    ftype=["Wind"],
    apply_remove_negatives=True,
    apply_remove_wind_zeros=True,
    apply_min_heatwave_days=True,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
)

Read gen_details & hw_tseries with Dask: 0.16 sec
Select group: 0.01 sec
--- Starting Dask-Native Process ---
Starting final Dask compute...
Dask compute finished.

--- DASK TIMING REPORT ---
duid_setup: 0.00 seconds
csv_bulk_read: 0.38 seconds
filter_and_clean: 0.01 seconds
type_conversion_and_dropna: 0.02 seconds
date_filter: 0.01 seconds
hw_tseries_filter: 12.06 seconds
aggregate_daily: 0.04 seconds
final_merge: 0.02 seconds
compute: 38.46 seconds
--- END REPORT ---

Process group: 56.18 sec

--- DEBUG: Calculated Heatwave Days per Generator ---
DUID
MUSSELR1    334
SAPHWF1     206
WRWF1       201
HALLWF2     187
HALLWF1     187
GUNNING1    178
WOODLWN1    171
LKBONNY2    167
GULLRWF1    164
NBHWF1      162
WATERLWF    161
CLEMGPWF    158
SNOWTWN1    157
LKBONNY3    156
BLUFF1      151
TARALGA1    150
MEWF1       143
BOCORWF1    126
BODWF1      122
STWF1       115
SNOWNTH1    109
OAKLAND1    107
COOPGWF1    103
BALDHWF1    101
MACARTH1     98
HDWF1        98
HDWF2        92
HDWF3   

In [4]:
df = data.copy()

df["year"] = pd.to_datetime(df["time"]).dt.year
first_year = (
    df.groupby("DUID", as_index=False)["year"]
      .min()
      .rename(columns={"year": "first_year"})
)

print(first_year)

        DUID  first_year
0      ARWF1        2016
1   BALDHWF1        2015
2     BLUFF1        2011
3   BOCORWF1        2014
4     BODWF1        2018
5   BULGANA1        2020
6   CATHROCK        2022
7    CHYTWF1        2020
8   CLEMGPWF        2009
9   CNUNDAWF        2021
10  COOPGWF1        2019
11  CROOKWF2        2018
12  CROWLWF1        2018
13   CRURWF1        2020
14   CTHLWF1        2020
15   DULAWF1        2023
16  ELAINWF1        2020
17   GRANWF1        2019
18  GULLRWF1        2013
19  GUNNING1        2011
20   HALLWF1        2009
21   HALLWF2        2009
22     HDWF1        2016
23     HDWF2        2017
24     HDWF3        2017
25  KABANWF1        2022
26    KEPWF1        2021
27  KIATAWF1        2017
28   LGAPWF1        2019
29   LGAPWF2        2021
30  LKBONNY1        2021
31  LKBONNY2        2009
32  LKBONNY3        2010
33  MACARTH1        2012
34  MERCER01        2013
35     MEWF1        2018
36  MTGELWF1        2018
37  MUSSELR1        2013
38   MUWAWF1        2019


In [6]:
counts = (
    df.groupby("DUID")["EHF_flag"]
      .value_counts()
      .unstack(fill_value=0)
      .rename(columns={0: "heatwave", 1: "baseline"})
      .reset_index()
)

print(counts)

EHF_flag      DUID  heatwave  baseline
0            ARWF1      2843        72
1         BALDHWF1      3340       101
2           BLUFF1      4594       151
3         BOCORWF1      3457       126
4           BODWF1      2057       122
5         BULGANA1      1469        28
6         CATHROCK       861        20
7          CHYTWF1      1498        27
8         CLEMGPWF      5292       158
9         CNUNDAWF       911        24
10        COOPGWF1      1751       103
11        CROOKWF2      2079        76
12        CROWLWF1      1988        63
13         CRURWF1      1274        41
14         CTHLWF1      1611        26
15         DULAWF1       415        39
16        ELAINWF1      1514        22
17         GRANWF1      1588        84
18        GULLRWF1      3710       164
19        GUNNING1      4665       178
20         HALLWF1      5279       187
21         HALLWF2      5279       187
22           HDWF1      2852        98
23           HDWF2      2595        92
24           HDWF3      2

In [7]:
details = first_year.merge(counts[['DUID','heatwave','baseline']], on='DUID')

In [11]:
details = details.merge(info[['station_name','region','DUID']], on='DUID')

In [12]:
details.to_csv("data/output/wind_chapter/wind_details_table.csv")